# HELIXOR MEDTECH — Distribution Operations Analysis

**Portfolio case study · Python/Pandas · April–September 2026**

This notebook analyzes a **simulated** 1,000-row medical-device distribution dataset. It covers data profiling, quality audit, safe cleaning, derived fields, provisional delivery KPIs, warehouse/route comparisons and an order-month pivot.

**How to reproduce:** Open in Google Colab and run the first cell. Upload `Raw_Data.csv` (download it from this repository first), or an Excel workbook (`.xlsx`/`.xls`) containing a `Raw_Data` sheet, then run the remaining cells from top to bottom. The source dataset is included in this repository at `data/Raw_Data.csv`. This notebook does not require Google Sheets authentication or claim approved operational KPIs.

**Important:** On-Time Delivery (OTD) and unit-weighted Quantity Fill Rate use **different subsets**. The reported KPI definitions and duplicate exclusions are provisional, not company-approved. Analysis of damaged units is confined to the 487 quantity-eligible shipment records.

**Reports:** [Full Technical Report](https://docs.google.com/document/d/1qdxnN2AF81fq0EMg8DhNVdsCub70UntLqm5gTTjE4y0/edit) · [Client-Ready Management Report](https://docs.google.com/document/d/1OLO-gvVqANnnAbInU6x6Hpj2X5kCxnUwjwy6gwGVqtk/edit)


In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("Belum ada file yang dipilih.")

nama_file = next(iter(uploaded))
print("File terpilih:", nama_file)

if nama_file.lower().endswith(".csv"):
    tabel_penerimaan = pd.read_csv(
        nama_file,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )
elif nama_file.lower().endswith((".xlsx", ".xls")):
    tabel_penerimaan = pd.read_excel(nama_file, sheet_name="Raw_Data")
else:
    raise ValueError("Pilih file CSV atau Excel.")

print("Total rows:", len(tabel_penerimaan))
print("Total columns:", len(tabel_penerimaan.columns))
display(tabel_penerimaan.head())

In [ ]:
print('DATA TYPE')
print()
print('---------------------------')

print(tabel_penerimaan.dtypes)

In [ ]:
print('MISSING DATA')
print()
print('---------------------------')
print(tabel_penerimaan.isna().sum())

In [ ]:
print(' TOTAL MISSING DATA')
print()
print('---------------------------')

print(tabel_penerimaan.isna().sum().sum())

In [ ]:
tabel_penerimaan.loc[
    tabel_penerimaan["warehouse"].isna(),
    ["shipment_id", "warehouse"]
]

In [ ]:
tabel_penerimaan.loc[
    tabel_penerimaan["promised_delivery_date"].isna(),
    ["shipment_id", "promised_delivery_date"]
]

In [ ]:
tabel_penerimaan.loc[
    tabel_penerimaan["actual_delivery_date"].isna(),
    ["shipment_id", "actual_delivery_date"]
]

In [ ]:
tabel_penerimaan.loc[
    tabel_penerimaan["quantity_delivered_units"].isna(),
    ["shipment_id", "quantity_delivered_units"]
]

In [ ]:
print(tabel_penerimaan.duplicated().sum())

In [ ]:
print(tabel_penerimaan.duplicated())

In [ ]:
display(
    tabel_penerimaan.loc[
        tabel_penerimaan.duplicated(),
        ["shipment_id"]
    ]
)

In [ ]:
display(
    tabel_penerimaan[tabel_penerimaan.duplicated(keep=False)]
)

In [ ]:
duplicated = tabel_penerimaan[
    tabel_penerimaan.duplicated(keep=False)
].copy()

duplicated["raw_data"] = duplicated.index

ringkasan_duplikat = (
    duplicated
    .groupby(list(tabel_penerimaan.columns), dropna=False)
    .agg(
        jumlah_kemunculan=("raw_data", "size"),
        index_baris=("raw_data", list)
    )
    .reset_index()
)

display(ringkasan_duplikat)

In [ ]:
print('INVALID VALUE')
print()
print('---------------------------')


quantity_numeric = pd.to_numeric(
    tabel_penerimaan["quantity_delivered_units"],
    errors="coerce"
)

invalid_quantity = (
    tabel_penerimaan["quantity_delivered_units"].notna()
    & quantity_numeric.isna()
)

display(
    tabel_penerimaan.loc[
        invalid_quantity,
        ["shipment_id", "quantity_delivered_units"]
    ]
)

In [ ]:
# 1. Konversi tanggal untuk pemeriksaan
tanggal_konversi = pd.to_datetime(
    tabel_penerimaan["promised_delivery_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi tanggal yang gagal dikonversi
invalid_date = (
    tabel_penerimaan["promised_delivery_date"].notna()
    & tanggal_konversi.isna()
)

# 3. Tampilkan shipment_id dan nilai tanggal bermasalah
display(
    tabel_penerimaan.loc[
        invalid_date,
        ["shipment_id", "promised_delivery_date"]
    ]
)

# 4. Hitung jumlah record yang bermasalah
print("Jumlah invalid date:", invalid_date.sum())

In [ ]:
# 1. Konversi tanggal untuk pemeriksaan
tanggal_konversi = pd.to_datetime(
    tabel_penerimaan["order_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi tanggal yang gagal dikonversi
invalid_date = (
    tabel_penerimaan["order_date"].notna()
    & tanggal_konversi.isna()
)

# 3. Tampilkan shipment_id dan nilai tanggal bermasalah
display(
    tabel_penerimaan.loc[
        invalid_date,
        ["shipment_id", "order_date"]
    ]
)

# 4. Hitung jumlah record yang bermasalah
print("Jumlah invalid date:", invalid_date.sum())

In [ ]:
print("Dtype sebelum:", tabel_penerimaan["order_date"].dtype)

print("Dtype hasil konversi:", tanggal_konversi.dtype)

print("Missing values:", tabel_penerimaan["order_date"].isna().sum())

print("Invalid dates:", invalid_date.sum())

In [ ]:
# 1. Konversi tanggal untuk pemeriksaan
tanggal_konversi = pd.to_datetime(
    tabel_penerimaan["dispatch_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi tanggal yang gagal dikonversi
invalid_date = (
    tabel_penerimaan["dispatch_date"].notna()
    & tanggal_konversi.isna()
)

# 3. Tampilkan shipment_id dan nilai tanggal bermasalah
display(
    tabel_penerimaan.loc[
        invalid_date,
        ["shipment_id", "dispatch_date"]
    ]
)

# 4. Hitung jumlah record yang bermasalah
print("Jumlah invalid date:", invalid_date.sum())

In [ ]:
# 1. Konversi tanggal untuk pemeriksaan
tanggal_konversi = pd.to_datetime(
    tabel_penerimaan["actual_delivery_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi tanggal yang gagal dikonversi
invalid_date = (
    tabel_penerimaan["actual_delivery_date"].notna()
    & tanggal_konversi.isna()
)

# 3. Tampilkan shipment_id dan nilai tanggal bermasalah
display(
    tabel_penerimaan.loc[
        invalid_date,
        ["shipment_id", "actual_delivery_date"]
    ]
)

# 4. Hitung jumlah record yang bermasalah
print("Jumlah invalid date:", invalid_date.sum())

In [ ]:
# Daftar warehouse yang diharapkan
valid_warehouse = [
    "Rotterdam",
    "Hamburg",
    "Frankfurt",
    "Lyon",
    "Madrid",
    "Milan"
]

# Identifikasi kategori yang tidak sesuai
invalid_warehouse = (
    tabel_penerimaan["warehouse"].notna()
    & ~tabel_penerimaan["warehouse"].isin(valid_warehouse)
)

# Tampilkan record yang perlu diperiksa
display(
    tabel_penerimaan.loc[
        invalid_warehouse,
        ["shipment_id", "warehouse"]
    ]
)

# Hitung jumlah record
print("Jumlah warehouse tidak sesuai:", invalid_warehouse.sum())

In [ ]:
tabel_penerimaan["destination_country"].unique()

In [ ]:
# Daftar negara yang diharapkan
valid_country = [
    "Austria",
    "France",
    "Italy",
    "United Kingdom",
    "Belgium",
    "Netherlands",
    "Germany",
    "Switzerland",
    "Sweden",
    "Spain",
    "Denmark"
]

# Identifikasi negara di luar daftar
invalid_country = (
    tabel_penerimaan["destination_country"].notna()
    & ~tabel_penerimaan["destination_country"].isin(valid_country)
)

# Tampilkan shipment yang perlu diperiksa
display(
    tabel_penerimaan.loc[
        invalid_country,
        ["shipment_id", "destination_country"]
    ]
)

# Hitung jumlah record
print("Jumlah negara tidak sesuai:", invalid_country.sum())

In [ ]:
# 1. Daftar kolom tanggal
kolom_tanggal = [
    "order_date",
    "promised_delivery_date",
    "dispatch_date",
    "actual_delivery_date"
]

# 2. Pemeriksaan setiap kolom
for kolom in kolom_tanggal:

    nilai = tabel_penerimaan[kolom].astype("string").str.strip()

    # Cek missing values
    missing = (
        nilai.isna()
        | nilai.eq("")
    ).fillna(False)

    # Percobaan konversi tanggal
    tanggal_konversi = pd.to_datetime(
        nilai,
        errors="coerce",
        format="mixed"
    )

    # Cek invalid dates (selain missing)
    invalid = (
        ~missing
        & tanggal_konversi.isna()
    )

    print(f"\n===== {kolom} =====")
    print("Missing values:", missing.sum())
    print("Invalid dates:", invalid.sum())

    # Tampilkan record missing
    if missing.any():
        print("\nMissing records:")
        display(
            tabel_penerimaan.loc[
                missing,
                ["shipment_id", "shipment_status", kolom]
            ]
        )

    # Tampilkan record invalid
    if invalid.any():
        print("\nInvalid records:")
        display(
            tabel_penerimaan.loc[
                invalid,
                ["shipment_id", "shipment_status", kolom]
            ]
        )

In [ ]:
# Daftar kategori sementara untuk pemeriksaan
valid_transport = [
    "Road",
    "Rail",
    "Air",
    "Courier"
]

# Identifikasi kategori tidak sesuai
invalid_transport = (
    tabel_penerimaan["transport_mode"].notna()
    & ~tabel_penerimaan["transport_mode"].isin(valid_transport)
)

# Tampilkan shipment yang perlu diperiksa
display(
    tabel_penerimaan.loc[
        invalid_transport,
        ["shipment_id", "transport_mode"]
    ]
)

# Hitung jumlah record
print("Jumlah transport_mode tidak sesuai:", invalid_transport.sum())

In [ ]:
print('NEGATIVE VALUE')
print()
print('---------------------------')

# 1. Konversi sementara untuk pemeriksaan
quantity_numeric = pd.to_numeric(
    tabel_penerimaan["quantity_ordered_units"],
    errors="coerce"
)

# 2. Identifikasi quantity negatif
negative_quantity = quantity_numeric < 0

# 3. Tampilkan record yang melanggar aturan
display(
    tabel_penerimaan.loc[
        negative_quantity,
        ["shipment_id", "quantity_ordered_units"]
    ]
)

# 4. Hitung jumlah pelanggaran
print(
    "Jumlah negative ordered quantity:",
    negative_quantity.sum()
)

In [ ]:
# Periksa nilai negatif pada quantity_delivered_units

delivered_numeric = pd.to_numeric(
    tabel_penerimaan["damaged_units"],
    errors="coerce"
)

negative_delivered = delivered_numeric < 0

print("Jumlah nilai negatif:", negative_delivered.sum())

display(
    tabel_penerimaan.loc[
        negative_delivered,
        ["shipment_id", "damaged_units"]
    ]
)

In [ ]:
print('Over delivery')
print()
print('---------------------------')

# 1. Konversi sementara ke numeric
ordered = pd.to_numeric(
    tabel_penerimaan["quantity_ordered_units"],
    errors="coerce"
)

delivered = pd.to_numeric(
    tabel_penerimaan["quantity_delivered_units"],
    errors="coerce"
)

# 2. Identifikasi overdelivery
overdelivery = (
    ordered.notna()
    & delivered.notna()
    & (ordered >= 0)
    & (delivered > ordered)
)

# 3. Tampilkan shipment yang bermasalah
display(
    tabel_penerimaan.loc[
        overdelivery,
        [
            "shipment_id",
            "quantity_ordered_units",
            "quantity_delivered_units"
        ]
    ]
)

# 4. Hitung jumlah record
print("Jumlah overdelivery:", overdelivery.sum())

In [ ]:
# 1. Konversi sementara menjadi numeric
delivered = pd.to_numeric(
    tabel_penerimaan["quantity_delivered_units"],
    errors="coerce"
)

damaged = pd.to_numeric(
    tabel_penerimaan["damaged_units"],
    errors="coerce"
)

# 2. Identifikasi pelanggaran business rule
invalid_damaged = (
    delivered.notna()
    & damaged.notna()
    & (damaged > delivered)
)

# 3. Tampilkan shipment yang bermasalah
display(
    tabel_penerimaan.loc[
        invalid_damaged,
        [
            "shipment_id",
            "quantity_delivered_units",
            "damaged_units"
        ]
    ]
)

# 4. Hitung jumlah pelanggaran
print(
    "Jumlah damaged units violation:",
    invalid_damaged.sum()
)

In [ ]:
# 1. Konversi sementara menjadi datetime
dispatch = pd.to_datetime(
    tabel_penerimaan["dispatch_date"],
    errors="coerce",
    format="mixed"
)

actual_delivery = pd.to_datetime(
    tabel_penerimaan["actual_delivery_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi pelanggaran business rule
invalid_delivery = (
    dispatch.notna()
    & actual_delivery.notna()
    & (actual_delivery < dispatch)
)

# 3. Tampilkan shipment yang bermasalah
display(
    tabel_penerimaan.loc[
        invalid_delivery,
        [
            "shipment_id",
            "shipment_status",
            "dispatch_date",
            "actual_delivery_date"
        ]
    ]
)

# 4. Hitung jumlah pelanggaran
print(
    "Jumlah delivery before dispatch:",
    invalid_delivery.sum()
)

In [ ]:
# 1. Konversi sementara menjadi datetime
order_date = pd.to_datetime(
    tabel_penerimaan["order_date"],
    errors="coerce",
    format="mixed"
)

dispatch_date = pd.to_datetime(
    tabel_penerimaan["dispatch_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi pelanggaran business rule
invalid_order_dispatch = (
    order_date.notna()
    & dispatch_date.notna()
    & (order_date > dispatch_date)
)

# 3. Tampilkan shipment yang bermasalah
display(
    tabel_penerimaan.loc[
        invalid_order_dispatch,
        [
            "shipment_id",
            "shipment_status",
            "order_date",
            "dispatch_date"
        ]
    ]
)

# 4. Hitung jumlah pelanggaran
print(
    "Jumlah order after dispatch:",
    invalid_order_dispatch.sum()
)

In [ ]:
# 1. Periksa tanggal penerimaan
actual_date = pd.to_datetime(
    tabel_penerimaan["actual_delivery_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi Delivered tanpa tanggal valid
invalid_delivered = (
    tabel_penerimaan["shipment_status"].eq("Delivered")
    & actual_date.isna()
)

# 3. Tampilkan record
display(
    tabel_penerimaan.loc[
        invalid_delivered,
        [
            "shipment_id",
            "shipment_status",
            "actual_delivery_date"
        ]
    ]
)

# 4. Hitung jumlah temuan
print(
    "Delivered tanpa tanggal valid:",
    invalid_delivered.sum()
)

In [ ]:
# 1. Konversi sementara tanggal penerimaan
actual_date = pd.to_datetime(
    tabel_penerimaan["actual_delivery_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi In Transit dengan tanggal penerimaan
invalid_in_transit = (
    tabel_penerimaan["shipment_status"].eq("In Transit")
    & actual_date.notna()
)

# 3. Tampilkan shipment yang perlu diperiksa
display(
    tabel_penerimaan.loc[
        invalid_in_transit,
        [
            "shipment_id",
            "shipment_status",
            "actual_delivery_date"
        ]
    ]
)

# 4. Hitung jumlah temuan
print(
    "Jumlah In Transit dengan tanggal penerimaan:",
    invalid_in_transit.sum()
)

In [ ]:
# 1. Konversi sementara tanggal penerimaan
actual_date = pd.to_datetime(
    tabel_penerimaan["actual_delivery_date"],
    errors="coerce",
    format="mixed"
)

# 2. Identifikasi Cancelled dengan tanggal penerimaan
invalid_cancelled = (
    tabel_penerimaan["shipment_status"].eq("Cancelled")
    & actual_date.notna()
)

# 3. Tampilkan shipment yang perlu diperiksa
display(
    tabel_penerimaan.loc[
        invalid_cancelled,
        [
            "shipment_id",
            "shipment_status",
            "actual_delivery_date"
        ]
    ]
)

# 4. Hitung jumlah temuan
print(
    "Jumlah Cancelled dengan tanggal penerimaan:",
    invalid_cancelled.sum()
)

In [ ]:
# 1. Konversi sementara quantity
delivered = pd.to_numeric(
    tabel_penerimaan["quantity_delivered_units"],
    errors="coerce"
)

# 2. Ambil shipment status
status = tabel_penerimaan["shipment_status"]

# 3. Pemeriksaan logical consistency
logical_issue = (
    ((status == "Delivered") & (delivered <= 0))
    |
    ((status == "Cancelled") & (delivered > 0))
    |
    ((status == "In Transit") & (delivered > 0))
    |
    ((status == "Partial") & (delivered <= 0))
)

# 4. Tampilkan shipment yang perlu investigasi
display(
    tabel_penerimaan.loc[
        logical_issue,
        [
            "shipment_id",
            "shipment_status",
            "quantity_ordered_units",
            "quantity_delivered_units"
        ]
    ]
)

# 5. Hitung jumlah temuan
print(
    "Jumlah potential logical inconsistencies:",
    logical_issue.sum()
)

In [ ]:
# 1. Konversi sementara menjadi numerik
quantity = pd.to_numeric(
    tabel_penerimaan["quantity_ordered_units"],
    errors="coerce"
)

# 2. Pisahkan nilai negatif yang sudah diaudit sebelumnya
quantity_valid = quantity[quantity >= 0]

# 3. Lihat distribusi data
print("Quantity Distribution:")
display(
    quantity_valid.describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# 4. Tampilkan 10 shipment dengan quantity terbesar
print("Top 10 Ordered Quantities:")

display(
    tabel_penerimaan.loc[
        quantity_valid.nlargest(10).index,
        ["shipment_id", "quantity_ordered_units"]
    ]
)

In [ ]:
# Hitung Q1 dan Q3
Q1 = quantity_valid.quantile(0.25)
Q3 = quantity_valid.quantile(0.75)

# Hitung IQR
IQR = Q3 - Q1

# Hitung batas outlier
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Tampilkan hasil
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

In [ ]:
# OA-02 — Quantity Delivered: Initial Inspection

delivered = pd.to_numeric(
    tabel_penerimaan["quantity_delivered_units"],
    errors="coerce"
)

delivered_valid = delivered[delivered >= 0]

# Ringkasan jumlah barang yang dikirim
display(
    delivered_valid.describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# 10 shipment dengan jumlah pengiriman terbesar
display(
    tabel_penerimaan.loc[
        delivered_valid.nlargest(10).index,
        ["shipment_id", "shipment_status", "quantity_delivered_units"]
    ]
)

In [ ]:
print("Total baris:", len(tabel_penerimaan))

print("Tidak terbaca sebagai angka:", delivered.isna().sum())

print("Nilai negatif:", (delivered < 0).sum())

In [ ]:
# Cari data yang tidak terbaca sebagai angka
tidak_terbaca = delivered.isna()

# Lihat 15 contoh
display(
    tabel_penerimaan.loc[
        tidak_terbaca,
        ["shipment_id", "shipment_status", "quantity_delivered_units"]
    ].head(15)
)

# Hitung berdasarkan status pengiriman
print(
    tabel_penerimaan.loc[
        tidak_terbaca, "shipment_status"
    ].value_counts(dropna=False)
)

In [ ]:
# OA-02 — Investigasi Delivered tanpa quantity yang valid

masalah_delivered = (
    (tabel_penerimaan["shipment_status"] == "Delivered")
    & delivered.isna()
)

display(
    tabel_penerimaan.loc[
        masalah_delivered,
        [
            "shipment_id",
            "shipment_status",
            "quantity_ordered_units",
            "quantity_delivered_units",
            "notes"
        ]
    ]
)

In [ ]:
# OA-02 — IQR Quantity Delivered

# Hitung Q1 dan Q3
Q1_delivered = delivered_valid.quantile(0.25)
Q3_delivered = delivered_valid.quantile(0.75)

# Hitung IQR
IQR_delivered = Q3_delivered - Q1_delivered

# Hitung batas outlier
lower_delivered = Q1_delivered - 1.5 * IQR_delivered
upper_delivered = Q3_delivered + 1.5 * IQR_delivered

# Tampilkan hasil
print("Q1:", Q1_delivered)
print("Q3:", Q3_delivered)
print("IQR:", IQR_delivered)
print("Lower Bound:", lower_delivered)
print("Upper Bound:", upper_delivered)

# Cari pengiriman yang melewati batas
outliers_delivered = delivered_valid[
    (delivered_valid < lower_delivered) |
    (delivered_valid > upper_delivered)
]

print("Jumlah Outlier:", len(outliers_delivered))

In [ ]:
# OA-03 — Damaged Units: Initial Inspection

damaged = pd.to_numeric(
    tabel_penerimaan["damaged_units"],
    errors="coerce"
)

damaged_valid = damaged[damaged >= 0]

print("Damaged Units Distribution:")

display(
    damaged_valid.describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("Jumlah nilai kosong/tidak terbaca:", damaged.isna().sum())
print("Jumlah nilai negatif:", (damaged < 0).sum())
print("Jumlah nilai nol:", (damaged == 0).sum())

# 10 shipment dengan jumlah barang rusak terbesar
display(
    tabel_penerimaan.loc[
        damaged_valid.nlargest(10).index,
        [
            "shipment_id",
            "shipment_status",
            "quantity_ordered_units",
            "quantity_delivered_units",
            "damaged_units",
            "quality_status",
            "notes"
        ]
    ]
)

In [ ]:
# OA-03 — IQR Damaged Units

# 1. Hitung Q1 dan Q3
Q1_damaged = damaged_valid.quantile(0.25)
Q3_damaged = damaged_valid.quantile(0.75)

# 2. Hitung IQR
IQR_damaged = Q3_damaged - Q1_damaged

# 3. Hitung batas outlier
lower_damaged = Q1_damaged - 1.5 * IQR_damaged
upper_damaged = Q3_damaged + 1.5 * IQR_damaged

# 4. Tampilkan hasil
print("Q1:", Q1_damaged)
print("Q3:", Q3_damaged)
print("IQR:", IQR_damaged)
print("Lower Bound:", lower_damaged)
print("Upper Bound:", upper_damaged)

# 5. Cari outlier
outliers_damaged = damaged_valid[
    (damaged_valid < lower_damaged) |
    (damaged_valid > upper_damaged)
]

print("Jumlah Outlier:", len(outliers_damaged))

# 6. Tampilkan shipment yang terdeteksi outlier
display(
    tabel_penerimaan.loc[
        outliers_damaged.index,
        [
            "shipment_id",
            "shipment_status",
            "quantity_ordered_units",
            "quantity_delivered_units",
            "damaged_units",
            "quality_status",
            "notes"
        ]
    ]
)

In [ ]:
# KR-01 — OTD Readiness
# Pemeriksaan kualitas data tanggal

date_columns = [
    "promised_delivery_date",
    "actual_delivery_date"
]

for col in date_columns:

    # Ambil nilai asli
    original = tabel_penerimaan[col].astype("string").str.strip()
    original = original.replace("", pd.NA)

    # Coba ubah menjadi tanggal
    converted = pd.to_datetime(
        original,
        errors="coerce",
        format="mixed"
    )

    # Hitung kondisi data
    valid = converted.notna().sum()
    missing = original.isna().sum()
    invalid = (original.notna() & converted.isna()).sum()

    print(f"\nKolom: {col}")
    print("Tanggal valid:", valid)
    print("Tanggal kosong:", missing)
    print("Tanggal tidak valid:", invalid)

    # Tampilkan contoh tanggal yang tidak valid
    display(
        tabel_penerimaan.loc[
            original.notna() & converted.isna(),
            ["shipment_id", "shipment_status", col]
        ]
    )

In [ ]:
# Cari pengiriman yang tanggal sampainya kosong
tanggal_kosong = (
    tabel_penerimaan["actual_delivery_date"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .isna()
)

# Hitung berdasarkan status pengiriman
print(
    tabel_penerimaan.loc[
        tanggal_kosong, "shipment_status"
    ].value_counts(dropna=False)
)

In [ ]:
# KR-01 — Investigasi tanggal kosong
# pada shipment Delivered dan Partial

status_perlu_cek = (
    tabel_penerimaan["shipment_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(["delivered", "partial"])
)

display(
    tabel_penerimaan.loc[
        tanggal_kosong & status_perlu_cek,
        [
            "shipment_id",
            "shipment_status",
            "promised_delivery_date",
            "actual_delivery_date",
            "notes"
        ]
    ]
)

In [ ]:
# KR-01 — Pemeriksaan awal kesiapan OTD

status = (
    tabel_penerimaan["shipment_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

janji = pd.to_datetime(
    tabel_penerimaan["promised_delivery_date"],
    errors="coerce",
    format="mixed"
)

sampai = pd.to_datetime(
    tabel_penerimaan["actual_delivery_date"],
    errors="coerce",
    format="mixed"
)

# Shipment Delivered dengan kedua tanggal valid
calon_otd = (
    (status == "delivered")
    & janji.notna()
    & sampai.notna()
)

# Shipment Delivered yang masih bermasalah tanggalnya
belum_siap = (
    (status == "delivered")
    & ~ (janji.notna() & sampai.notna())
)

print("Total Delivered:", (status == "delivered").sum())
print("Calon siap OTD:", calon_otd.sum())
print("Belum siap (masalah tanggal):", belum_siap.sum())

In [ ]:
# KR-01 — Lihat shipment yang belum siap OTD

display(
    tabel_penerimaan.loc[
        belum_siap,
        [
            "shipment_id",
            "shipment_status",
            "promised_delivery_date",
            "actual_delivery_date",
            "notes"
        ]
    ]
)

## Data Quality Findings & Business Impact


In [ ]:
# KR-01 — Periksa urutan tanggal pengiriman

dispatch = pd.to_datetime(
    tabel_penerimaan["dispatch_date"],
    errors="coerce",
    format="mixed"
)

# Cari shipment yang tanggal sampainya lebih awal dari keberangkatan
tanggal_tidak_logis = (
    calon_otd
    & dispatch.notna()
    & (sampai < dispatch)
)

print("Jumlah shipment dengan urutan tanggal tidak logis:")
print(tanggal_tidak_logis.sum())

display(
    tabel_penerimaan.loc[
        tanggal_tidak_logis,
        [
            "shipment_id",
            "shipment_status",
            "dispatch_date",
            "actual_delivery_date",
            "notes"
        ]
    ]
)

In [ ]:
# KR-02 — Periksa quantity pada shipment Delivered

status_delivered = (
    tabel_penerimaan["shipment_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    == "delivered"
)

ordered = pd.to_numeric(
    tabel_penerimaan["quantity_ordered_units"],
    errors="coerce"
)

delivered_qty = pd.to_numeric(
    tabel_penerimaan["quantity_delivered_units"],
    errors="coerce"
)

masalah_quantity = (
    status_delivered
    & (
        ordered.isna()
        | (ordered <= 0)
        | delivered_qty.isna()
        | (delivered_qty <= 0)
        | (delivered_qty > ordered)
    )
)

print("Jumlah shipment Delivered dengan masalah quantity:")
print(masalah_quantity.sum())

display(
    tabel_penerimaan.loc[
        masalah_quantity,
        [
            "shipment_id",
            "shipment_status",
            "quantity_ordered_units",
            "quantity_delivered_units",
            "notes"
        ]
    ]
)

In [ ]:
df_raw = tabel_penerimaan.copy()
df_clean = df_raw.copy()

In [ ]:
df_clean["warehouse"] = (
    df_clean["warehouse"]
    .str.strip()
    .replace({
        "rotterdam": "Rotterdam",
        "ROTTERDAM": "Rotterdam",
        "lyon": "Lyon"
    })
)

In [ ]:
changed = (
    df_raw["warehouse"].fillna("__MISSING__")
    != df_clean["warehouse"].fillna("__MISSING__")
)

comparison = pd.DataFrame({
    "shipment_id": df_raw.loc[changed, "shipment_id"],
    "Before": df_raw.loc[changed, "warehouse"],
    "After": df_clean.loc[changed, "warehouse"]
})

display(comparison)

print("Total changed:", changed.sum())
print("Row count unchanged:", len(df_raw) == len(df_clean))

In [ ]:
df_clean["promised_delivery_date_dt"] = pd.to_datetime(
    df_clean["promised_delivery_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
print(df_clean["promised_delivery_date_dt"].dtype)

print(
    df_clean["promised_delivery_date_dt"]
    .isna()
    .sum()
)

In [ ]:
df_clean["actual_delivery_date_dt"] = pd.to_datetime(
    df_clean["actual_delivery_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
print(df_clean["actual_delivery_date_dt"].dtype)
print(df_clean["actual_delivery_date_dt"].isna().sum())

In [ ]:
df_clean["dispatch_date_dt"] = pd.to_datetime(
    df_clean["dispatch_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
df_clean["order_date_dt"] = pd.to_datetime(
    df_clean["order_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [ ]:
print(df_clean["order_date_dt"].dtype)
print("Unconverted dates:", df_clean["order_date_dt"].isna().sum())

In [ ]:
df_clean.loc[
    df_clean["order_date_dt"].isna(),
    ["shipment_id", "order_date"]
]

In [ ]:
day_part = pd.to_numeric(
    df_clean["order_date"].astype("string").str.split("/").str[0],
    errors="coerce"
)

safe_mask = df_clean["order_date_dt"].isna() & (day_part > 12)

df_clean.loc[safe_mask, "order_date_dt"] = pd.to_datetime(
    df_clean.loc[safe_mask, "order_date"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Converted:", safe_mask.sum())
print("Remaining NaT:", df_clean["order_date_dt"].isna().sum())

In [ ]:
df_clean["quantity_delivered_numeric"] = pd.to_numeric(
    df_clean["quantity_delivered_units"],
    errors="coerce"
)

In [ ]:
print(df_clean["quantity_delivered_numeric"].dtype)

print(
    "Total NaN:",
    df_clean["quantity_delivered_numeric"].isna().sum()
)

In [ ]:
print(df_clean[[
    "quantity_ordered_units",
    "damaged_units"
]].dtypes)

In [ ]:
print("=== FINAL CLEANING VALIDATION ===")

print("Rows unchanged:", len(df_raw) == len(df_clean))

print("Warehouse corrected:", changed.sum())

print("Promised date NaT:", df_clean["promised_delivery_date_dt"].isna().sum())

print("Actual date NaT:", df_clean["actual_delivery_date_dt"].isna().sum())

print("Dispatch date NaT:", df_clean["dispatch_date_dt"].isna().sum())

print("Order date NaT:", df_clean["order_date_dt"].isna().sum())

print("Delivered quantity NaN:", df_clean["quantity_delivered_numeric"].isna().sum())

In [ ]:
original_columns = df_raw.columns.drop("warehouse")

unchanged = df_raw[original_columns].equals(
    df_clean[original_columns]
)

print("Other original columns unchanged:", unchanged)

In [ ]:
df_clean["quantity_short_units"] = (
    df_clean["quantity_ordered_units"]
    - df_clean["quantity_delivered_numeric"]
)

In [ ]:
display(df_clean[[
    "shipment_id",
    "shipment_status",
    "quantity_ordered_units",
    "quantity_delivered_numeric",
    "quantity_short_units"
]].head(10))

In [ ]:
df_clean["delivery_delay_days"] = (
    df_clean["actual_delivery_date_dt"]
    - df_clean["promised_delivery_date_dt"]
).dt.days

In [ ]:
display(df_clean[[
    "shipment_id",
    "shipment_status",
    "promised_delivery_date_dt",
    "actual_delivery_date_dt",
    "delivery_delay_days"
]].head(10))

**Preliminary Finding — Delivery Delay & Quantity Shortfall**

During an initial review of 10 shipment records, two Delivered shipments were identified with both delivery delays and quantity shortfalls:

- SHP-26-00204: 3 days late, 1 unit short.
- SHP-26-00059: 5 days late, 7 units short.

These findings suggest potential operational performance issues requiring further investigation. The underlying causes have not yet been confirmed.

**Status:** Preliminary observation — pending full-dataset analysis.

In [ ]:
print("=== FEATURE VALIDATION ===")

print(
    "Quantity calculable:",
    df_clean["quantity_short_units"].notna().sum()
)

print(
    "Delivery delay calculable:",
    df_clean["delivery_delay_days"].notna().sum()
)

print(
    "Both calculable:",
    (
        df_clean["quantity_short_units"].notna()
        & df_clean["delivery_delay_days"].notna()
    ).sum()
)

In [ ]:
print("Total raw rows:", len(df_raw))
print("Total clean rows:", len(df_clean))
print("Unique shipment IDs:", df_clean["shipment_id"].nunique())

In [ ]:
status_key = (
    df_clean["shipment_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

status_map = {
    "delivered": "Delivered",
    "partial": "Partial",
    "in transit": "In Transit",
    "cancelled": "Cancelled"
}

df_clean["shipment_status_std"] = (
    status_key
    .map(status_map)
    .fillna(df_clean["shipment_status"])
)

print(df_clean["shipment_status_std"].value_counts(dropna=False))

In [ ]:
# Date availability for provisional OTD (normalized shipment status)
valid_dates = (
    df_clean["promised_delivery_date_dt"].notna()
    & df_clean["actual_delivery_date_dt"].notna()
)

delivered_std = (
    df_clean["shipment_status_std"] == "Delivered"
)

otd_candidates = df_clean[
    delivered_std & valid_dates
].copy()

print("Total Delivered:", delivered_std.sum())
print("Delivered with valid dates:", len(otd_candidates))

In [ ]:
duplicate_mask = otd_candidates.duplicated(
    subset=df_raw.columns.tolist(),
    keep=False
)

date_issue = (
    otd_candidates["actual_delivery_date_dt"]
    < otd_candidates["dispatch_date_dt"]
)

otd_working = otd_candidates.loc[
    ~(duplicate_mask | date_issue)
].copy()

on_time = (otd_working["delivery_delay_days"] <= 0).sum()
late = (otd_working["delivery_delay_days"] > 0).sum()
total = len(otd_working)

otd_rate = (on_time / total) * 100

print("Duplicate rows flagged:", duplicate_mask.sum())
print("Date issues flagged:", date_issue.sum())
print("Total evaluated:", total)
print("On-time:", on_time)
print("Late:", late)
print(f"OTD Rate: {otd_rate:.2f}%")

## Provisional Quantity Fill Rate


In [ ]:
fill_candidates = df_clean.loc[
    (df_clean["shipment_status_std"] == "Delivered")
    & (df_clean["quantity_ordered_units"] > 0)
    & df_clean["quantity_delivered_numeric"].notna()
].copy()

fill_duplicates = fill_candidates.duplicated(
    subset=df_raw.columns.tolist(),
    keep=False
)

overdelivery = (
    fill_candidates["quantity_delivered_numeric"]
    > fill_candidates["quantity_ordered_units"]
)

zero_delivery = (
    fill_candidates["quantity_delivered_numeric"] == 0
)

print("Total candidates:", len(fill_candidates))
print("Duplicate rows flagged:", fill_duplicates.sum())
print("Overdelivery flagged:", overdelivery.sum())
print("Zero delivered flagged:", zero_delivery.sum())

In [ ]:
issue_count = (
    fill_duplicates.astype(int)
    + overdelivery.astype(int)
    + zero_delivery.astype(int)
)

print("Rows with multiple issues:", (issue_count > 1).sum())

print(
    "Candidates without these issues:",
    (issue_count == 0).sum()
)

In [ ]:
fill_working = fill_candidates.loc[
    ~(fill_duplicates | overdelivery | zero_delivery)
].copy()

total_ordered = fill_working["quantity_ordered_units"].sum()

total_delivered = fill_working["quantity_delivered_numeric"].sum()

fill_rate = (total_delivered / total_ordered) * 100

print("Total shipments evaluated:", len(fill_working))
print("Total units ordered:", total_ordered)
print("Total units delivered:", total_delivered)
print(f"Quantity Fill Rate: {fill_rate:.2f}%")

In [ ]:
warehouse_otd = (
    otd_working
    .assign(on_time=otd_working["delivery_delay_days"] <= 0)
    .groupby("warehouse", dropna=False)
    .agg(
        total_shipments=("shipment_id", "size"),
        on_time_shipments=("on_time", "sum")
    )
)

warehouse_otd["otd_rate"] = (
    warehouse_otd["on_time_shipments"]
    / warehouse_otd["total_shipments"]
    * 100
)

display(warehouse_otd.round(2))

In [ ]:
warehouse_otd["late_shipments"] = (
    warehouse_otd["total_shipments"]
    - warehouse_otd["on_time_shipments"]
)

display(
    warehouse_otd.sort_values("otd_rate").round(2)
)

In [ ]:
route_otd = (
    otd_working
    .assign(on_time=otd_working["delivery_delay_days"] <= 0)
    .groupby("route_id", dropna=False)
    .agg(
        total_shipments=("shipment_id", "size"),
        on_time_shipments=("on_time", "sum")
    )
)

route_otd["late_shipments"] = (
    route_otd["total_shipments"]
    - route_otd["on_time_shipments"]
)

route_otd["otd_rate"] = (
    route_otd["on_time_shipments"]
    / route_otd["total_shipments"]
    * 100
)

display(route_otd.sort_values("otd_rate").round(2))

In [ ]:
madrid_otd = otd_working[
    otd_working["warehouse"] == "Madrid"
].copy()

madrid_routes = (
    madrid_otd
    .assign(on_time=madrid_otd["delivery_delay_days"] <= 0)
    .groupby("route_id", dropna=False)
    .agg(
        total_shipments=("shipment_id", "size"),
        on_time_shipments=("on_time", "sum")
    )
)

madrid_routes["late_shipments"] = (
    madrid_routes["total_shipments"]
    - madrid_routes["on_time_shipments"]
)

madrid_routes["otd_rate"] = (
    madrid_routes["on_time_shipments"]
    / madrid_routes["total_shipments"] * 100
)

display(
    madrid_routes.sort_values("total_shipments", ascending=False).round(2)
)

In [ ]:
madrid_late = otd_working[
    (otd_working["warehouse"] == "Madrid")
    & (otd_working["delivery_delay_days"] > 0)
]

late_dispatch = (
    madrid_late["dispatch_date_dt"]
    > madrid_late["promised_delivery_date_dt"]
)

print("Madrid late shipments:", len(madrid_late))
print("Dispatched after promised date:", late_dispatch.sum())

In [ ]:
madrid_transit = madrid_late.loc[~late_dispatch].copy()

madrid_transit["travel_days"] = (
    madrid_transit["actual_delivery_date_dt"]
    - madrid_transit["dispatch_date_dt"]
).dt.days

madrid_transit["available_days"] = (
    madrid_transit["promised_delivery_date_dt"]
    - madrid_transit["dispatch_date_dt"]
).dt.days

display(madrid_transit[[
    "shipment_id",
    "available_days",
    "travel_days",
    "delivery_delay_days"
]].head(10))

In [ ]:
display(
    madrid_transit[[
        "available_days",
        "travel_days",
        "delivery_delay_days"
    ]].agg(["count", "mean", "median"]).round(2)
)

In [ ]:
warehouse_timing = otd_working.copy()

warehouse_timing["available_days"] = (
    warehouse_timing["promised_delivery_date_dt"]
    - warehouse_timing["dispatch_date_dt"]
).dt.days

warehouse_timing["travel_days"] = (
    warehouse_timing["actual_delivery_date_dt"]
    - warehouse_timing["dispatch_date_dt"]
).dt.days

comparison = warehouse_timing.groupby("warehouse").agg(
    shipments=("shipment_id", "size"),
    median_available=("available_days", "median"),
    median_travel=("travel_days", "median")
)

display(comparison)

In [ ]:
pivot_data = otd_working.copy()

pivot_data["order_month"] = (
    pivot_data["order_date_dt"]
    .dt.to_period("M")
)

In [ ]:
display(pivot_data[[
    "shipment_id",
    "warehouse",
    "order_date_dt",
    "order_month"
]].head(10))

print(
    "Order month unavailable:",
    pivot_data["order_month"].isna().sum()
)

In [ ]:
pivot_ready = pivot_data.dropna(
    subset=["order_month"]
).copy()

pivot_ready["warehouse_group"] = (
    pivot_ready["warehouse"].fillna("Unknown Warehouse")
)

pivot_ready["on_time"] = (
    pivot_ready["delivery_delay_days"] <= 0
)

otd_pivot = (
    pivot_ready.pivot_table(
        index="warehouse_group",
        columns="order_month",
        values="on_time",
        aggfunc="mean"
    ) * 100
)

display(otd_pivot.round(2))

print("Rows included:", len(pivot_ready))

In [ ]:
shipment_count_pivot = pivot_ready.pivot_table(
    index="warehouse_group",
    columns="order_month",
    values="shipment_id",
    aggfunc="count",
    fill_value=0
)

display(shipment_count_pivot)

print("Total shipments:", shipment_count_pivot.to_numpy().sum())

In [ ]:

warehouse_fill = (
    fill_working
    .groupby("warehouse", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        ordered_units=("quantity_ordered_units", "sum"),
        delivered_units=("quantity_delivered_numeric", "sum")
    )
)

warehouse_fill["fill_rate_pct"] = (
    warehouse_fill["delivered_units"]
    / warehouse_fill["ordered_units"]
    * 100
)

display(warehouse_fill.sort_values("fill_rate_pct").round(2))

print("Total shipments:", warehouse_fill["shipments"].sum())
print("Total ordered:", warehouse_fill["ordered_units"].sum())
print("Total delivered:", warehouse_fill["delivered_units"].sum())


In [ ]:

route_fill = (
    fill_working
    .groupby("route_id", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        ordered_units=("quantity_ordered_units", "sum"),
        delivered_units=("quantity_delivered_numeric", "sum")
    )
)

route_fill["fill_rate_pct"] = (
    route_fill["delivered_units"]
    / route_fill["ordered_units"]
    * 100
)

display(route_fill.sort_values("fill_rate_pct").round(2))

print("Total shipments:", route_fill["shipments"].sum())
print("Total ordered:", route_fill["ordered_units"].sum())
print("Total delivered:", route_fill["delivered_units"].sum())


In [ ]:

warehouse_damage = (
    fill_working
    .groupby("warehouse", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        delivered_units=("quantity_delivered_numeric", "sum"),
        damaged_units=("damaged_units", "sum")
    )
)

display(
    warehouse_damage.sort_values(
        "damaged_units",
        ascending=False
    )
)

print("Total shipments:", warehouse_damage["shipments"].sum())
print("Total damaged units:", warehouse_damage["damaged_units"].sum())


In [ ]:
route_damage = (
    fill_working
    .groupby("route_id", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        delivered_units=("quantity_delivered_numeric", "sum"),
        damaged_units=("damaged_units", "sum")
    )
)

display(
    route_damage.sort_values(
        "damaged_units",
        ascending=False
    )
)

print("Total shipments:", route_damage["shipments"].sum())
print("Total damaged units:", route_damage["damaged_units"].sum())

In [ ]:
route_warehouse_damage = (
    fill_working[fill_working["route_id"] == "HAM-DE-03"]
    .groupby("warehouse", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        damaged_units=("damaged_units", "sum")
    )
    .sort_values("damaged_units", ascending=False)
)

display(route_warehouse_damage)

print("Total damaged:", route_warehouse_damage["damaged_units"].sum())

In [ ]:
route_warehouse_damage = (
    fill_working[fill_working["route_id"] == "FRA-SE-03"]
    .groupby("warehouse", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        damaged_units=("damaged_units", "sum")
    )
    .sort_values("damaged_units", ascending=False)
)

display(route_warehouse_damage)

print("Total damaged:", route_warehouse_damage["damaged_units"].sum())

In [ ]:
route_warehouse_damage = (
    fill_working[fill_working["route_id"] == "HAM-DK-02"]
    .groupby("warehouse", dropna=False)
    .agg(
        shipments=("shipment_id", "size"),
        damaged_units=("damaged_units", "sum")
    )
    .sort_values("damaged_units", ascending=False)
)

display(route_warehouse_damage)

print("Total damaged:", route_warehouse_damage["damaged_units"].sum())

## Reproducibility and provisional KPI cross-check

Checks compare the calculations against the supplied training dataset. If the source changes, investigate the difference instead of adjusting results to match these expected values. All KPI figures remain subject to approval of eligibility and duplicate handling.


In [ ]:
# Provisional KPI cross-check against the training dataset
assert len(df_raw) == len(df_clean) == 1000
assert len(otd_working) == 476
assert (otd_working["delivery_delay_days"] <= 0).sum() == 110
assert (otd_working["delivery_delay_days"] > 0).sum() == 366
assert len(fill_working) == 487
assert fill_working["quantity_ordered_units"].sum() == 111679
assert fill_working["quantity_delivered_numeric"].sum() == 108864
assert fill_working["damaged_units"].sum() == 1001
assert warehouse_fill["shipments"].sum() == route_fill["shipments"].sum() == 487
assert warehouse_damage["damaged_units"].sum() == route_damage["damaged_units"].sum() == 1001
assert len(pivot_ready) == 474

print("All provisional cross-checks passed for this training dataset.")
print("OTD: 110 / 476 = 23.11%")
print("Quantity Fill Rate: 108,864 / 111,679 = 97.48%")
print("Recorded damaged units: 1,001 / 108,864 = 0.92%")
